# Phase III Code Verification

This notebook starts Phase III with a manufactured-solution grid refinement study. It runs `main_solver.py` at `Re = 10` on systematically refined meshes, computes the discretization error norms, and calculates the observed order of accuracy so the solution can be checked against second-order convergence.

In [3]:
from pathlib import Path
import contextlib
import io
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Run from coarsest to finest mesh so the quickest cases finish first.
MESH_STUDY = [
    {"mesh": 5, "nodes": 9, "h": 16},
    {"mesh": 4, "nodes": 17, "h": 8},
    {"mesh": 3, "nodes": 33, "h": 4},
    {"mesh": 2, "nodes": 65, "h": 2},
    {"mesh": 1, "nodes": 129, "h": 1},
]
REYNOLDS_NUMBER = 10.0
EQUATIONS = ["p", "u", "v"]


def apply_replacements(source, replacements, solver_path):
    for old, new in replacements.items():
        if old not in source:
            raise RuntimeError(f"Could not find expected setting in {solver_path}: {old}")
        source = source.replace(old, new, 1)
    return source


def run_mms_solver(nodes):
    """Run main_solver.py for one MMS grid and return its error norms."""
    solver_path = Path("main_solver.py")
    source = solver_path.read_text()
    replacements = {
        "imax = 9               # Number of points in the x-direction (use odd numbers only)":
            f"imax = {nodes:<15}# Number of points in the x-direction (use odd numbers only)",
        "jmax = 9               # Number of points in the y-direction (use odd numbers only)":
            f"jmax = {nodes:<15}# Number of points in the y-direction (use odd numbers only)",
        "imms = 0               # Manufactured solution flag: 1 for manuf. sol., 0 otherwise":
            "imms = 1               # Manufactured solution flag: 1 for manuf. sol., 0 otherwise",
        "Re = 10.0              # Reynolds number = rho*Uinf*L/rmu":
            f"Re = {REYNOLDS_NUMBER:<16g}# Reynolds number = rho*Uinf*L/rmu",
        "vectorize = False":
            "vectorize = True",
    }
    source = apply_replacements(source, replacements, solver_path)

    namespace = {
        "__file__": str(solver_path.resolve()),
        "__name__": "__main__",
    }

    # main_solver.py prints every run and creates figures; keep this study output compact.
    with contextlib.redirect_stdout(io.StringIO()) as captured_output:
        exec(compile(source, str(solver_path), "exec"), namespace)
    plt.close("all")

    return {
        "rL1norm": namespace["rL1norm"].copy(),
        "rL2norm": namespace["rL2norm"].copy(),
        "rLinfnorm": namespace["rLinfnorm"].copy(),
        "iterations": namespace["n"],
        "converged": namespace["isConverged"],
        "final_residual": namespace["res"].copy(),
        "captured_output": captured_output.getvalue(),
    }


def build_error_table(results):
    rows = []
    for case in results:
        for eq_idx, equation in enumerate(EQUATIONS):
            rows.append({
                "mesh": case["mesh"],
                "nodes": f"{case['nodes']}x{case['nodes']}",
                "h": case["h"],
                "equation": equation,
                "L1": case["rL1norm"][eq_idx],
                "L2": case["rL2norm"][eq_idx],
                "Linf": case["rLinfnorm"][eq_idx],
                "iterations": case["iterations"],
                "converged": case["converged"],
            })
    return pd.DataFrame(rows)


def observed_order(error_table, norm_name="L2"):
    order_rows = []
    for equation in EQUATIONS:
        eq_table = error_table[error_table["equation"] == equation].sort_values("h", ascending=False)
        previous = None
        for _, row in eq_table.iterrows():
            if previous is not None:
                order = np.log(previous[norm_name] / row[norm_name]) / np.log(previous["h"] / row["h"])
                order_rows.append({
                    "equation": equation,
                    "coarse_mesh": int(previous["mesh"]),
                    "fine_mesh": int(row["mesh"]),
                    "coarse_h": previous["h"],
                    "fine_h": row["h"],
                    f"observed_order_{norm_name}": order,
                })
            previous = row
    return pd.DataFrame(order_rows)


if Path.cwd().name != "start-code" and (Path.cwd() / "start-code").exists():
    os.chdir(Path.cwd() / "start-code")

print(f"Working directory: {Path.cwd()}")
print(f"Running MMS verification study at Re = {REYNOLDS_NUMBER:g} from coarsest to finest mesh")

results = []
for case in MESH_STUDY:
    print(f"Running mesh {case['mesh']}: {case['nodes']}x{case['nodes']} nodes, h = {case['h']}")
    results.append({**case, **run_mms_solver(case["nodes"])})

error_table = build_error_table(results)
order_table = observed_order(error_table, norm_name="L2")

print("\nDiscretization error norms:")
display(error_table)

print("\nObserved order of accuracy from L2 norms:")
display(order_table)

print("\nFine-grid observed orders should approach 2 for a second-order method.")

Working directory: /Users/windy/mae4100-courseproject2/start-code
Running MMS verification study at Re = 10 from coarsest to finest mesh
Running mesh 5: 9x9 nodes, h = 16
Running mesh 4: 17x17 nodes, h = 8
Running mesh 3: 33x33 nodes, h = 4
Running mesh 2: 65x65 nodes, h = 2
Running mesh 1: 129x129 nodes, h = 1

Discretization error norms:


,mesh,nodes,h,equation,L1,L2,Linf,iterations,converged
0,5,9x9,16,p,1.595031e-03,2.129081e-03,4.507980e-03,436,True
1,5,9x9,16,u,4.081134e-04,6.585967e-04,1.744505e-03,436,True
2,5,9x9,16,v,1.528813e-04,1.779708e-04,3.841467e-04,436,True
3,4,17x17,8,p,2.550080e-04,3.614914e-04,1.071097e-03,1633,True
4,4,17x17,8,u,4.486766e-05,8.123910e-05,3.055250e-04,1633,True
5,4,17x17,8,v,2.651298e-05,3.068330e-05,6.227238e-05,1633,True
6,3,33x33,4,p,5.380738e-05,7.362721e-05,2.709283e-04,6282,True
7,3,33x33,4,u,4.625998e-06,9.598730e-06,5.023628e-05,6282,True
8,3,33x33,4,v,5.210913e-06,6.066099e-06,1.157085e-05,6282,True
9,2,65x65,2,p,1.273450e-05,1.655326e-05,7.111572e-05,24425,True



Observed order of accuracy from L2 norms:


,equation,coarse_mesh,fine_mesh,coarse_h,fine_h,observed_order_L2
0,p,5,4,16,8,2.558198
1,p,4,3,8,4,2.295650
2,p,3,2,4,2,2.153124
3,p,2,1,2,1,2.070127
4,u,5,4,16,8,3.019149
5,u,4,3,8,4,3.081259
6,u,3,2,4,2,2.920157
7,u,2,1,2,1,2.221000
8,v,5,4,16,8,2.536115
9,v,4,3,8,4,2.338613



Fine-grid observed orders should approach 2 for a second-order method.


In [ ]:
# L2 error vs mesh spacing (log-log) with second-order reference
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=False, sharey=False)
for ax, eq in zip(axes, EQUATIONS):
    sub = error_table[error_table["equation"] == eq].sort_values("h")
    h_vals = sub["h"].to_numpy(dtype=float)
    e_l2 = sub["L2"].to_numpy(dtype=float)
    ax.loglog(h_vals, e_l2, "o-", color="C0", label=r"$||e||_{L^2}$")
    # Reference: ||e|| ∝ h^2, anchored at finest mesh (smallest h)
    h_line = np.array([h_vals.min(), h_vals.max()])
    e_ref = e_l2[-1] * (h_line / h_vals[-1]) ** 2
    ax.loglog(h_line, e_ref, "k--", lw=1.5, label=r"slope $= 2$")
    ax.set_xlabel(r"$h$")
    ax.set_ylabel(r"$||e||_{L^2}$")
    ax.set_title(f"equation `{eq}`")
    ax.legend(loc="best")
    ax.grid(True, which="both", ls=":", alpha=0.7)
plt.suptitle(r"MMS grid refinement: $||e||_{L^2}$ vs.\ $h$ (log--log)")
plt.tight_layout()
plt.show()